# Lab 05: White-Box Native AutoML (Gemini + Scikit-Learn)
**[🇧🇷 PT-BR]** AutoML Nativo White-Box (Predição de Churn & Explicabilidade SHAP/Feature Importance)  
**[🇺🇸 EN]** White-Box Native AutoML (Customer Churn Prediction & Feature Importance Explainability)  
**Course / Disciplina:** Augmented Analytics & AI-Driven Insights (FIAP MBA)  
**Official Engine / Modelo Oficial:** Gemini 3.8 Flash (`gemini-3.8-flash`) via `google-genai` SDK

---

* 🇧🇷 Este notebook demonstra como criar um agente autônomo de Machine Learning que projeta, escreve, executa e valida seu próprio pipeline de predição de churn (*White-Box AutoML*) com Scikit-Learn.
* 🇺🇸 This notebook demonstrates how to build an autonomous Machine Learning agent that designs, writes, executes, and validates its own customer churn prediction pipeline (*White-Box AutoML*) using Scikit-Learn.

In [ ]:
# 1. Install official Google GenAI SDK, Excel reader, and utilities
!pip install -q -U google-genai openpyxl tabulate

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google import genai
from google.genai import types
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import re

# 2. Configure API Key & Output Language
# 🇧🇷 Substitua 'COLE_SUA_API_KEY_AQUI' pela sua chave do Google AI Studio | 🇺🇸 Replace with your Google AI Studio API Key
API_KEY = 'COLE_SUA_API_KEY_AQUI'
OUTPUT_LANGUAGE = 'Portuguese (Brazil)' # @param ['Portuguese (Brazil)', 'English']
client = genai.Client(api_key=API_KEY)

In [ ]:
# 3. Resilient Self-Healing Churn Dataset Ingestion
import os
import urllib.request

DATASET_NAME = 'Lab 05 - EvasaoClientesClassificados.xlsx'
RAW_URL = 'https://raw.githubusercontent.com/rafaelmatsuyama/FIAP-BI-AugmentedAnalytics/main/lab05-automl-scikit/Lab%2005%20-%20EvasaoClientesClassificados.xlsx'

if not os.path.exists(DATASET_NAME):
    print(f"📥 Downloading / Baixando {DATASET_NAME} from official repository...")
    urllib.request.urlretrieve(RAW_URL, DATASET_NAME)

try:
    df = pd.read_excel(DATASET_NAME)
    print("✅ Dataset 'Lab 05 - EvasaoClientesClassificados.xlsx' loaded successfully! / carregado com sucesso!")
    print(f"Total customers analyzed / Total de clientes analisados: {len(df)}")
    display(df.head(3))

    # Prepare schema and sample context for the AI Data Scientist agent
    contexto_colunas = df.dtypes.to_string()
    amostra_dados = df.head(3).to_markdown()
except Exception as e:
    print(f"❌ Error loading Excel dataset / Erro ao carregar o arquivo Excel: {e}")

In [ ]:
# 4. NATIVE WHITE-BOX AUTOML ENGINE (Autonomous Data Scientist Agent)
def executar_automl_nativo(pergunta_negocio="Crie um modelo preditivo de churn"):
    """
    Uses Gemini 3.8 Flash to design, write, and execute an end-to-end
    White-Box Machine Learning pipeline in Scikit-Learn.
    """
    prompt = f"""
    You are a Senior Data Scientist specialized in Machine Learning and Scikit-Learn.
    The dataframe 'df' is already loaded in memory with the following schema:
    {contexto_colunas}

    Data sample:
    {amostra_dados}

    The target column to predict is 'Perdido?'.

    YOUR TASK:
    Write a complete, self-contained Python script that:
    1. Drops unique identifiers or non-predictive columns (such as 'ID Cliente').
    2. Encodes categorical variables (such as 'Sexo', 'Região', 'Nivel Cobertura') using LabelEncoder or get_dummies.
    3. Splits the data into train and test sets (80% train / 20% test) with random_state=42.
    4. Trains a robust classification model (RandomForestClassifier by default, with n_estimators=100).
    5. Prints the Accuracy and the full Classification Report (classification_report).
    6. Computes and plots the Confusion Matrix and the Feature Importance bar chart using matplotlib/seaborn.
    7. OBJECTIVELY ANSWERS THE BUSINESS QUESTION IN {OUTPUT_LANGUAGE}: {pergunta_negocio}

    STRICT RULES:
    - Return ONLY raw executable Python code enclosed inside a ```python block.
    - Do not include introductory or explanatory text outside the code block.
    - Ensure 'plt.show()' is called to render the charts.
    """

    try:
        print(f"🤖 AutoML Agent: Designing and training predictive pipeline ({OUTPUT_LANGUAGE})...\n")
        response = client.models.generate_content(
            model='gemini-3.8-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.2,
                top_p=0.3
            )
        )

        match = re.search(r'```python\n(.*?)\n```', response.text, re.DOTALL)
        if match:
            codigo = match.group(1).strip()
            print("⚡ Code synthesized by Gemini 3.8 Flash! Executing training pipeline...\n")
            exec(codigo, globals())
            print("\n✅ AutoML pipeline executed and validated successfully! / Pipeline executado com sucesso!")
        else:
            print("❌ Error: Could not extract Python code block from model response.")
            print(response.text)
    except Exception as e:
        print(f"⚠️ AutoML execution error: {e}")
        if 'codigo' in locals():
            print(f"Attempted code / Código tentado:\n{codigo}")

# International English alias
run_native_automl = executar_automl_nativo

In [ ]:
# 5. AUTOML EXECUTION: Churn Driver Discovery / Descoberta de Drivers de Churn
executar_automl_nativo("Identifique as 3 variáveis que mais influenciam a saída do cliente e avalie a qualidade do modelo.")
# EN Example: run_native_automl("Identify the top 3 features driving customer attrition and evaluate model performance.")

In [ ]:
# 6. SENSITIVITY ANALYSIS: Coverage Tier Impact / Análise de Nível de Cobertura
executar_automl_nativo("O nível de cobertura influencia diretamente no Churn? Mostre a correlação nos dados.")
# EN Example: run_native_automl("Does coverage tier directly impact customer churn? Show the correlation in the data.")